# Week 2 Day 5 — Multimodal Agent (Anthropic edition)

> **Targets Gradio 6.** History flattened via `to_text()`; Blocks UI used (no `type="messages"`).

Claude doesn't generate images or speech, so: **audio → gTTS** (free, no key), **images → optional** (default no-op; HuggingFace option provided). The agent/tool core is pure Claude.

In [3]:
import os
import json
from dotenv import load_dotenv
import anthropic
import gradio as gr
import sqlite3

load_dotenv(override=True)
print("Anthropic key set" if os.getenv("ANTHROPIC_API_KEY") else "Anthropic key NOT set")

MODEL = "claude-haiku-4-5"
client = anthropic.Anthropic()
DB = "prices.db"

Anthropic key set


In [4]:
def to_text(content):
    """Gradio 6 gives message content as a list of blocks; older Gradio gave a plain string.
    Anthropic wants a plain string here, so flatten it. Handles both versions safely."""
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text", "")
    if isinstance(content, list):
        return "".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text")
    return str(content)

In [5]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [10]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        row = conn.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),)).fetchone()
        return f"Ticket price to {city} is ${row[0]}" if row else "No price data available for this city"

get_ticket_price("Mumbai")

DATABASE TOOL CALLED: Getting price for Mumbai


'Ticket price to Mumbai is $1600.0'

In [7]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "input_schema": {
        "type": "object",
        "properties": {
            "destination_city": {"type": "string", "description": "The city the customer wants to travel to"},
        },
        "required": ["destination_city"],
    },
}
tools = [price_function]

## Audio — gTTS (free substitute for OpenAI TTS). `pip install gtts`

In [9]:
from gtts import gTTS
import tempfile

def talker(message):
    tts = gTTS(text=message, lang="en")
    path = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3").name
    tts.save(path)
    return path

## Image — optional (no Anthropic equivalent). Default no-op so the app runs.

In [11]:
def artist(city):
    return None   # Anthropic can't generate images

# --- OPTIONAL free images via HuggingFace (needs free HF_TOKEN in .env) ---
# from huggingface_hub import InferenceClient
# def artist(city):
#     hf = InferenceClient(token=os.getenv("HF_TOKEN"))
#     return hf.text_to_image(f"A vibrant pop-art travel poster of {city}",
#                             model="stabilityai/stable-diffusion-xl-base-1.0")

## The agent loop

In [12]:
def handle_tool_calls_and_return_cities(response):
    results, cities = [], []
    for block in response.content:
        if block.type == "tool_use" and block.name == "get_ticket_price":
            city = block.input.get("destination_city")
            cities.append(city)
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": get_ticket_price(city)})
    return results, cities

def text_from(response):
    return "".join(b.text for b in response.content if b.type == "text")

In [13]:
def chat(history):
    # Flatten Gradio-6 history into Anthropic-friendly messages
    messages = [{"role": h["role"], "content": to_text(h["content"])} for h in history]
    response = client.messages.create(model=MODEL, max_tokens=500, system=system_message, messages=messages, tools=tools)
    cities = []

    while response.stop_reason == "tool_use":
        tool_results, new_cities = handle_tool_calls_and_return_cities(response)
        cities += new_cities
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
        response = client.messages.create(model=MODEL, max_tokens=500, system=system_message, messages=messages, tools=tools)

    reply = text_from(response)
    history = list(history) + [{"role": "assistant", "content": reply}]
    voice = talker(reply)
    image = artist(cities[0]) if cities else None
    return history, voice, image

## Custom UI with Gradio Blocks

In [15]:
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)   # note: gr.Chatbot still takes type=
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## Note
`gr.Chatbot(type="messages")` is still valid in Gradio 6 — only `gr.ChatInterface` dropped the `type` argument. Different components, different rules.

## Exercises
Add a booking tool; apply the pattern to your own domain. For media, the production shape is Claude-for-reasoning + a separate specialist model called as a tool.